In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/jobs.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

In [ ]:
print("Columns in the dataset:")
print(df.columns.tolist())

In [ ]:
print("Dataset information:")
df.info()

In [ ]:
print("Fraudulent vs Legitimate Jobs:")
print(df["fraudulent"].value_counts())

In [ ]:
print("Fraudulent vs Legitimate Percentage:")
print(df["fraudulent"].value_counts(normalize=True) * 100)

In [ ]:
print("Number of rows and columns:")
print(df.shape)

In [ ]:
print("Columns in the dataset:\n")

for i, column in enumerate(df.columns, start=1):
    print(i, "→", column)

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum()

print("Missing values in each column:\n")
print(missing_values)

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print("Missing percentage:\n")
print(missing_percentage.sort_values(ascending=False))

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
pd.set_option("display.max_columns", None)

df.head(3)

In [ ]:
print(df["fraudulent"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

df["fraudulent"].value_counts().plot(kind="bar")

plt.title("Legitimate vs Fraudulent Job Postings")
plt.xlabel("Job Type")
plt.ylabel("Number of Job Postings")
plt.xticks([0, 1], ["Legitimate", "Fraudulent"], rotation=0)
plt.show()

In [ ]:
print(df.head())
print(df.shape)
print(df.info())
print(df["fraudulent"].value_counts())

In [ ]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nFraudulent Job Distribution:")
print(df["fraudulent"].value_counts())

In [ ]:
df = df.drop_duplicates()

print("Dataset Shape after removing duplicates:", df.shape)

In [ ]:
text_columns = [
    "location",
    "department",
    "salary_range",
    "company_profile",
    "requirements",
    "benefits",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

for column in text_columns:
    df[column] = df[column].fillna("Not Specified")

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

In [ ]:
print("\nDataset Shape:", df.shape)

print("\nFraudulent Job Distribution:")
print(df["fraudulent"].value_counts())

In [ ]:
df["fraudulent"] = df["fraudulent"].map({
    "f": 0,
    "t": 1
})

print(df["fraudulent"].value_counts())

In [ ]:
print(df["fraudulent"].unique())
print(df["fraudulent"].dtype)

In [ ]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

df["job_text"] = df[text_columns].fillna("").agg(" ".join, axis=1)

print(df[["title", "job_text", "fraudulent"]].head())

In [ ]:
print("Number of job postings:", len(df))

print("\nExample Job Text:")
print(df["job_text"].iloc[0])

In [ ]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.lower().strip()

df["clean_job_text"] = df["job_text"].apply(clean_text)

print("Original Text:")
print(df["job_text"].iloc[0][:500])

print("\nCleaned Text:")
print(df["clean_job_text"].iloc[0][:500])

In [ ]:
print("Empty job texts:", (df["clean_job_text"].str.len() == 0).sum())

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_job_text"]
y = df["fraudulent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Convert job text into numerical features
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Learn vocabulary from training data only
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

print("\nSample features:")
print(tfidf.get_feature_names_out()[:20])


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Convert job text into numerical features
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Learn vocabulary from training data only
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

print("\nSample features:")
print(tfidf.get_feature_names_out()[:20])


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Create the machine learning model
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train the model
model.fit(X_train_tfidf, y_train)

# Predict on unseen test data
y_pred = model.predict(X_test_tfidf)

# Evaluate the model
print("Model Training Completed!")

print("\nAccuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Fraudulent"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create the Naive Bayes model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train_tfidf, y_train)

# Make predictions
nb_pred = nb_model.predict(X_test_tfidf)

# Evaluate the model
print("Naive Bayes Model Evaluation")

print("\nAccuracy:",
      round(accuracy_score(y_test, nb_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    nb_pred,
    target_names=["Legitimate", "Fraudulent"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, nb_pred))


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Compare both trained models
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Fraud Precision", "Fraud Recall", "Fraud F1-Score"],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label=1),
        recall_score(y_test, y_pred, pos_label=1),
        f1_score(y_test, y_pred, pos_label=1)
    ],
    "Naive Bayes": [
        accuracy_score(y_test, nb_pred),
        precision_score(y_test, nb_pred, pos_label=1),
        recall_score(y_test, nb_pred, pos_label=1),
        f1_score(y_test, nb_pred, pos_label=1)
    ]
})

# Display comparison
print("\nMODEL COMPARISON")
print(comparison.round(3).to_string(index=False))


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Create a validation set from the training data
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train_tfidf,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# Train a model for threshold testing
threshold_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

threshold_model.fit(X_subtrain, y_subtrain)

# Get fraud probabilities on validation data
val_probabilities = threshold_model.predict_proba(X_val)[:, 1]

# Compare different thresholds
print("THRESHOLD COMPARISON\n")

for threshold in [0.30, 0.40, 0.50, 0.60]:
    val_pred = (val_probabilities >= threshold).astype(int)

    print(f"Threshold: {threshold}")
    print(f"Accuracy:  {accuracy_score(y_val, val_pred):.3f}")
    print(f"Precision: {precision_score(y_val, val_pred, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(y_val, val_pred, zero_division=0):.3f}")
    print(f"F1-Score:  {f1_score(y_val, val_pred, zero_division=0):.3f}")
    print("-" * 30)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Train the final model on all training data
final_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

final_model.fit(X_train_tfidf, y_train)

# Get fraud probabilities for the test set
test_probabilities = final_model.predict_proba(X_test_tfidf)[:, 1]

# Apply the selected threshold
threshold = 0.60
final_pred = (test_probabilities >= threshold).astype(int)

# Evaluate the final model
print("JOBSHIELD AI - FINAL MODEL RESULTS")
print("Selected threshold:", threshold)

print("\nAccuracy:",
      round(accuracy_score(y_test, final_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    final_pred,
    target_names=["Legitimate", "Fraudulent"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))


In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(final_model, "models/jobshield_model.pkl")
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
joblib.dump(threshold, "models/threshold.pkl")

print("JobShield AI model saved successfully!")
print("Files saved inside the models folder.")

In [ ]:
%pip install streamlit

In [ ]:
%pip show streamlit

In [ ]:
from pathlib import Path

project_folder = Path.cwd()
print("Your current folder is:", project_folder)
print("Files and folders:", list(project_folder.iterdir()))

In [ ]:
from pathlib import Path

models_folder = Path.cwd() / "models"

for file in models_folder.iterdir():
    print(file.name)

In [ ]:

from pathlib import Path

project_folder = Path.cwd().parent
app_path = project_folder / "app.py"

app_code = r'''
import streamlit as st
import joblib
import re
from pathlib import Path
from bs4 import BeautifulSoup

BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "notebooks" / "models"

model = joblib.load(MODEL_DIR / "jobshield_model.pkl")
tfidf = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
threshold = joblib.load(MODEL_DIR / "threshold.pkl")

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.lower().strip()

st.set_page_config(
    page_title="JobShield AI",
    page_icon="🛡️",
    layout="centered"
)

st.title("🛡️ JobShield AI")
st.subheader("AI-Powered Fake Job Detection")
st.write(
    "Analyze a job posting to identify signs of potential fraud."
)

job_description = st.text_area(
    "Paste the job description here:",
    height=250,
    placeholder="Paste the complete job posting..."
)

if st.button("Analyze Job", type="primary"):
    if not job_description.strip():
        st.warning("Please enter a job description first.")
    else:
        cleaned_text = clean_text(job_description)
        job_vector = tfidf.transform([cleaned_text])

        fraud_probability = model.predict_proba(job_vector)[0][1]
        prediction = int(fraud_probability >= threshold)

        st.divider()
        st.subheader("Analysis Result")
        st.metric("Estimated Fraud Risk", f"{fraud_probability * 100:.2f}%")

        if prediction == 1:
            st.error("⚠️ Potentially Fraudulent Job")
        else:
            st.success("✅ Likely Legitimate Job")

        st.caption(
            "This result is an AI estimate, not proof of fraud. "
            "Verify the employer and never pay money to secure a job."
        )
'''

app_path.write_text(app_code, encoding="utf-8")

print("App file created successfully!")
print("Location:", app_path)
print("Exists:", app_path.exists())


In [ ]:
import subprocess
import sys
from pathlib import Path

project_folder = Path.cwd().parent
app_path = project_folder / "app.py"

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(app_path),
        "--server.headless",
        "true"
    ],
    cwd=str(project_folder)
)

print("JobShield AI launch command started.")
print("Process ID:", process.pid)

In [ ]:
from pathlib import Path

project_folder = Path.cwd().parent
app_path = project_folder / "app.py"

app_code = r'''
import streamlit as st
import joblib
import re
from pathlib import Path
from bs4 import BeautifulSoup

# Load saved machine learning model and supporting files
BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "notebooks" / "models"

model = joblib.load(MODEL_DIR / "jobshield_model.pkl")
tfidf = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
threshold = float(joblib.load(MODEL_DIR / "threshold.pkl"))

# Clean the job posting text
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text.lower().strip()

# Page configuration
st.set_page_config(
    page_title="JobShield AI",
    page_icon="🛡️",
    layout="wide"
)

# Application heading
st.title("🛡️ JobShield AI")
st.subheader("AI-Powered Fake Job Detection")
st.write(
    "Screen job postings for potential fraud using machine learning."
)

st.info(
    "Enter the available information about the job posting. "
    "You can leave optional fields blank."
)

# Job information form
with st.form("job_analysis_form"):

    st.markdown("### 1. Job Information")

    title = st.text_input("Job Title")

    description = st.text_area(
        "Job Description *",
        height=200,
        placeholder="Paste the complete job description here..."
    )

    st.markdown("### 2. Company and Requirements")

    company_profile = st.text_area(
        "Company Profile",
        placeholder="Company background, website information, etc."
    )

    requirements = st.text_area(
        "Candidate Requirements",
        placeholder="Skills, qualifications, and experience required..."
    )

    benefits = st.text_area(
        "Salary, Benefits or Perks",
        placeholder="Salary, incentives, benefits, or other details..."
    )

    col1, col2 = st.columns(2)

    with col1:
        employment_type = st.text_input("Employment Type")
        experience = st.text_input("Required Experience")

    with col2:
        education = st.text_input("Required Education")
        industry = st.text_input("Industry")

    function = st.text_input("Job Function")

    submitted = st.form_submit_button(
        "Analyze Job Posting",
        type="primary",
        use_container_width=True
    )

# Analyze the submitted job
if submitted:

    if not description.strip():
        st.warning("Please enter the job description.")

    else:
        fields = [
            title,
            company_profile,
            description,
            requirements,
            benefits,
            employment_type,
            experience,
            education,
            industry,
            function
        ]

        combined_text = " ".join(
            value.strip()
            for value in fields
            if value.strip()
        )

        cleaned_text = clean_text(combined_text)

        job_vector = tfidf.transform([cleaned_text])

        fraud_probability = float(
            model.predict_proba(job_vector)[0][1]
        )

        prediction = int(fraud_probability >= threshold)

        st.divider()
        st.markdown("## Analysis Results")

        result_col, score_col = st.columns(2)

        with result_col:

            if prediction == 1:
                st.error("⚠️ Potentially Fraudulent")
                st.write(
                    "The model's estimated fraud score meets or "
                    "exceeds its decision threshold."
                )

            else:
                st.success("✅ Likely Legitimate")
                st.write(
                    "The estimated fraud score is below the decision "
                    "threshold. This does not guarantee the job is safe."
                )

        with score_col:

            st.metric(
                "Estimated Fraud Risk Score",
                f"{fraud_probability * 100:.2f}%"
            )

            st.progress(
                max(0.0, min(1.0, fraud_probability))
            )

            st.caption(
                f"Decision threshold: {threshold:.2f}"
            )

        st.markdown("### Before You Apply")

        st.markdown("""
        - Verify the employer through its official website.
        - Be cautious of upfront registration or recruitment fees.
        - Never share banking passwords or OTPs with recruiters.
        - Confirm that the recruiter and vacancy are genuine.
        """)

        st.caption(
            "This score is an uncalibrated model estimate, not a "
            "verified probability or proof of fraud. Independently "
            "verify the job before making a decision."
        )
'''

# Save the application file
app_path.write_text(app_code, encoding="utf-8")

print("Updated app.py successfully!")
print("File:", app_path)
print("Exists:", app_path.exists())

In [ ]:
import subprocess
import sys
from pathlib import Path

project_folder = Path.cwd().parent
app_path = project_folder / "app.py"

# Stop the previous app if its process is available
try:
    process.terminate()
    process.wait(timeout=5)
    print("Previous app stopped.")
except (NameError, Exception):
    print("No previous app process to stop.")

# Launch the updated app
process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(app_path),
        "--server.headless",
        "true"
    ],
    cwd=str(project_folder)
)

print("JobShield AI is launching!")
print("Open this address in your browser: http://localhost:8501")
print("Process ID:", process.pid)

In [ ]:
from pathlib import Path

project = Path.cwd().parent

print("PROJECT:")
print(project)

print("\nROOT APP:")
print(project / "app.py")
print("Exists:", (project / "app.py").exists())

print("\nAPP FOLDER APP:")
print(project / "app" / "app.py")
print("Exists:", (project / "app" / "app.py").exists())

print("\nROOT MODELS:")
for f in (project / "models").glob("*"):
    print(f.name)

print("\nNOTEBOOK MODELS:")
for f in (project / "notebooks" / "models").glob("*"):
    print(f.name)

In [ ]:
from pathlib import Path
import shutil

project = Path.cwd().parent

source = project / "notebooks" / "models"
destination = project / "models"

destination.mkdir(exist_ok=True)

for file in source.glob("*.pkl"):
    shutil.copy2(file, destination / file.name)
    print(f"Copied: {file.name}")

print("\nModels now available in:")
print(destination)

In [ ]:
from pathlib import Path

project = Path.cwd().parent

root_app = project / "app.py"
folder_app = project / "app" / "app.py"

print("ROOT app.py size:", root_app.stat().st_size, "bytes")
print("APP/app.py size:", folder_app.stat().st_size, "bytes")

print("\n--- ROOT app.py: first 20 lines ---")
print("\n".join(root_app.read_text(encoding="utf-8").splitlines()[:20]))

print("\n--- app/app.py: first 20 lines ---")
print("\n".join(folder_app.read_text(encoding="utf-8").splitlines()[:20]))

In [ ]:
from pathlib import Path

project = Path.cwd().parent
models_folder = project / "models"

for filename in [
    "jobshield_model.pkl",
    "tfidf_vectorizer.pkl",
    "threshold.pkl"
]:
    file_path = models_folder / filename
    print(f"{filename}: {'Found' if file_path.exists() else 'MISSING'}")

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No changes reported.")
if result.stderr:
    print(result.stderr)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "init"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short", "--ignored"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)

In [ ]:
from pathlib import Path

project = Path.cwd().parent
dataset = project / "data" / "raw" / "jobs.csv"

if dataset.exists():
    size_mb = dataset.stat().st_size / (1024 * 1024)
    print(f"Dataset size: {size_mb:.2f} MB")
else:
    print("Dataset file not found.")

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short", "--untracked-files=all"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "add", ".gitignore", "README.md", "app.py",
     "models", "notebooks/01_data_exploration.ipynb",
     "requirements.txt", "src"],
    cwd=project,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Project files staged successfully!")
else:
    print(result.stderr)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)

In [ ]:
import subprocess
from pathlib import Path

project = Path.cwd().parent

# Stage the latest notebook changes
subprocess.run(
    ["git", "add", "notebooks/01_data_exploration.ipynb"],
    cwd=project,
    check=True
)

# Create the first commit
result = subprocess.run(
    [
        "git", "commit",
        "-m", "Initial commit: JobShield AI fake job detection system"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("First commit created successfully!")

In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    [
        "git", "remote", "add", "origin",
        "https://github.com/sirshti-tech/JobShield.git"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("GitHub repository connected successfully!")
else:
    print(result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "push", "-u", "origin", "master"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("JobShield AI uploaded to GitHub successfully!")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

# Stage the updated README
subprocess.run(
    ["git", "add", "README.md"],
    cwd=project,
    check=True
)

# Commit the change
result = subprocess.run(
    ["git", "commit", "-m", "Update project structure in README"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "push"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("README update pushed to GitHub successfully!")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

# Add the updated README and screenshot
subprocess.run(
    ["git", "add", "README.md", "screenshots/jobshield-app.png"],
    cwd=project,
    check=True
)

# Commit the changes
result = subprocess.run(
    ["git", "commit", "-m", "Add application screenshot to README"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "push"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("Screenshot uploaded to GitHub successfully!")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

# Stage the updated notebook
subprocess.run(
    ["git", "add", "notebooks/01_data_exploration.ipynb"],
    cwd=project,
    check=True
)

# Commit the changes
result = subprocess.run(
    [
        "git", "commit",
        "-m", "Update data exploration notebook"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "push"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("Notebook update pushed to GitHub successfully!")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "diff", "--stat", "--", "notebooks/01_data_exploration.ipynb"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout or "No differences detected.")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

subprocess.run(
    ["git", "add", "notebooks/01_data_exploration.ipynb"],
    cwd=project,
    check=True
)

result = subprocess.run(
    ["git", "commit", "-m", "Save latest notebook updates"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "push"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("Latest notebook changes pushed successfully!")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project,
    capture_output=True,
    text=True
)

if result.returncode == 0 and not result.stdout.strip():
    print("Everything is synced. Working tree is clean!")
else:
    print(result.stdout or result.stderr)


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    [
        "git", "diff", "--numstat",
        "--", "notebooks/01_data_exploration.ipynb"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout or "No unstaged differences found.")


In [ ]:

import subprocess
from pathlib import Path

project = Path.cwd().parent

result = subprocess.run(
    [
        "git", "diff", "--",
        "notebooks/01_data_exploration.ipynb"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout[:5000])
